# Setup (Google Colab)

1. Upload `caption_data` to Google Drive at `MyDrive/DeepL-Final/Data/caption_data/` (folder must contain `captions.txt` and `Images/`).
2. Or upload `caption_data.zip` to `MyDrive/DeepL-Final/Data/` and run the next cell — it will unzip automatically.
3. Runtime → Change runtime type → **GPU** (recommended).

In [ ]:
# Environment & paths (Colab + local)
import zipfile
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/DeepL-Final')
else:
    PROJECT_ROOT = Path('.').resolve()

DATA_DIR = PROJECT_ROOT / 'Data' / 'caption_data'
IMAGES_DIR = DATA_DIR / 'Images'
CAPTIONS_FILE = DATA_DIR / 'captions.txt'
MODEL_DIR = PROJECT_ROOT / 'models'
DATA_ZIP = PROJECT_ROOT / 'Data' / 'caption_data.zip'

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not CAPTIONS_FILE.exists() and DATA_ZIP.exists():
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(PROJECT_ROOT / 'Data')

assert CAPTIONS_FILE.exists(), f'Missing captions file: {CAPTIONS_FILE}'
assert IMAGES_DIR.exists(), f'Missing images directory: {IMAGES_DIR}'

print('Project root:', PROJECT_ROOT)
print('Captions file:', CAPTIONS_FILE)
print('Model directory:', MODEL_DIR)


In [ ]:
import json
import math
import os
import random
import re
import sys
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Model architecture ---
ENCODER_NAME = 'resnet50'
ENCODER_DIM = 2048
EMBED_DIM = 256
HIDDEN_DIM = 512
DROPOUT = 0.5

# --- Data ---
IMAGE_SIZE = 224
MAX_SEQ_LEN = 40
MIN_WORD_FREQ = 5
TRAIN_RATIO, VAL_RATIO = 0.8, 0.1

# --- Training ---
BATCH_SIZE = 32 if torch.cuda.is_available() else 16
NUM_EPOCHS = 30
ENCODER_LR = 1e-5
DECODER_LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 5.0
FREEZE_ENCODER_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 6
USE_AMP = torch.cuda.is_available()

# 0 on Windows, Colab, and Python 3.14+ (forkserver pickling issues in notebooks)
NUM_WORKERS = 0 if (os.name == 'nt' or sys.version_info >= (3, 14)) else min(4, os.cpu_count() or 2)

START_TOKEN = '<start>'
END_TOKEN = '<end>'
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print('Device:', DEVICE)
print('Batch size:', BATCH_SIZE)
print('Mixed precision:', USE_AMP)


# Data Loading

In [ ]:
def tokenize_caption(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()


def build_vocabulary(captions, min_freq=MIN_WORD_FREQ):
    counter = Counter()
    for caption in captions:
        counter.update(tokenize_caption(caption))
    words = sorted(word for word, freq in counter.items() if freq >= min_freq)
    special = [PAD_TOKEN, START_TOKEN, END_TOKEN, UNK_TOKEN]
    vocab = special + words
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word


def encode_caption(caption, word2idx, max_len=MAX_SEQ_LEN):
    tokens = [START_TOKEN] + tokenize_caption(caption) + [END_TOKEN]
    unk_idx = word2idx[UNK_TOKEN]
    pad_idx = word2idx[PAD_TOKEN]
    indices = [word2idx.get(token, unk_idx) for token in tokens]
    if len(indices) > max_len:
        indices = indices[: max_len - 1] + [word2idx[END_TOKEN]]
    length = len(indices)
    indices += [pad_idx] * (max_len - len(indices))
    return torch.tensor(indices, dtype=torch.long), length


caption_df = pd.read_csv(CAPTIONS_FILE)
caption_df.columns = [c.strip() for c in caption_df.columns]
caption_df = caption_df.dropna(subset=['image', 'caption']).reset_index(drop=True)
caption_df['image'] = caption_df['image'].astype(str).str.strip()
caption_df['caption'] = caption_df['caption'].astype(str).str.strip()

image_names = sorted(caption_df['image'].unique())
rng = random.Random(SEED)
rng.shuffle(image_names)

n_total = len(image_names)
n_train = int(TRAIN_RATIO * n_total)
n_val = int(VAL_RATIO * n_total)

train_images = set(image_names[:n_train])
val_images = set(image_names[n_train : n_train + n_val])
test_images = set(image_names[n_train + n_val :])

train_df = caption_df[caption_df['image'].isin(train_images)].reset_index(drop=True)
val_df = caption_df[caption_df['image'].isin(val_images)].reset_index(drop=True)
test_df = caption_df[caption_df['image'].isin(test_images)].reset_index(drop=True)

word2idx, idx2word = build_vocabulary(train_df['caption'].tolist())
VOCAB_SIZE = len(word2idx)
PAD_IDX = word2idx[PAD_TOKEN]

caption_lengths = [len(tokenize_caption(c)) + 2 for c in train_df['caption']]

print(f'Images: train={len(train_images)}, val={len(val_images)}, test={len(test_images)}')
print(f'Captions: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
print(f'Vocabulary size: {VOCAB_SIZE} (min_freq={MIN_WORD_FREQ})')
print(
    f'Caption length (train): min={min(caption_lengths)}, '
    f'max={max(caption_lengths)}, p95={np.percentile(caption_lengths, 95):.0f}'
)


In [ ]:
sample_imgs = caption_df['image'].unique()[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, img_name in zip(axes, sample_imgs):
    img_path = IMAGES_DIR / img_name
    captions = caption_df[caption_df['image'] == img_name]['caption'].tolist()
    ax.imshow(Image.open(img_path).convert('RGB'))
    ax.set_title('\n'.join(captions[:2]), fontsize=7, wrap=True)
    ax.axis('off')
plt.suptitle('Sample images with reference captions', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE + 32),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class CaptionDataset(Dataset):
    def __init__(self, dataframe, images_dir, word2idx, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.word2idx = word2idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.images_dir / row['image']
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        caption_tensor, caption_length = encode_caption(row['caption'], self.word2idx)
        return image, caption_tensor, caption_length


def make_loader(dataframe, transform, shuffle):
    dataset = CaptionDataset(dataframe, IMAGES_DIR, word2idx, transform)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )


train_loader = make_loader(train_df, train_transform, shuffle=True)
val_loader = make_loader(val_df, eval_transform, shuffle=False)
test_loader = make_loader(test_df, eval_transform, shuffle=False)

images, captions, lengths = next(iter(train_loader))
print('Batch image shape:', images.shape)
print('Batch caption shape:', captions.shape)
print('Example token length:', lengths[0].item())
print(f'Train batches: {len(train_loader)} ({len(train_df)} caption rows, all references per image)')


# Model Definition

In [ ]:
class AttentionCaptionModel(nn.Module):
    """ResNet-50 encoder + Bahdanau attention + LSTM decoder."""

    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        encoder_dim,
        pad_idx,
        dropout=0.5,
        encoder_name='resnet50',
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.hidden_dim = hidden_dim
        self.encoder_dim = encoder_dim

        if encoder_name == 'resnet50':
            backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        elif encoder_name == 'resnet18':
            backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            encoder_dim = 512
            self.encoder_dim = 512
        else:
            raise ValueError(f'Unsupported encoder: {encoder_name}')

        self.encoder = nn.Sequential(*list(backbone.children())[:-2])
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))

        self.init_h = nn.Linear(encoder_dim, hidden_dim)
        self.init_c = nn.Linear(encoder_dim, hidden_dim)

        self.attention_enc = nn.Linear(encoder_dim, hidden_dim, bias=False)
        self.attention_dec = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attention_full = nn.Linear(hidden_dim, 1)

        self.context_proj = nn.Linear(encoder_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm_cell = nn.LSTMCell(embed_dim + hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

        self._init_decoder_weights()

    def _init_decoder_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        nn.init.uniform_(self.fc.weight, -0.1, 0.1)
        nn.init.constant_(self.fc.bias, 0)

    def encode_image(self, images):
        features = self.encoder(images)
        features = self.adaptive_pool(features)
        batch_size, channels, height, width = features.shape
        features = features.view(batch_size, channels, height * width).permute(0, 2, 1)
        mean_features = features.mean(dim=1)
        h = torch.tanh(self.init_h(mean_features))
        c = torch.tanh(self.init_c(mean_features))
        return features, h, c

    def attention(self, encoder_out, hidden):
        enc = self.attention_enc(encoder_out)
        dec = self.attention_dec(hidden).unsqueeze(1)
        energy = torch.tanh(enc + dec)
        weights = torch.softmax(self.attention_full(energy).squeeze(-1), dim=1)
        context = (encoder_out * weights.unsqueeze(-1)).sum(dim=1)
        return self.context_proj(context), weights

    def decode_step(self, encoder_out, prev_tokens, hidden, cell):
        embeddings = self.dropout(self.embedding(prev_tokens))
        context, _ = self.attention(encoder_out, hidden)
        lstm_input = torch.cat([embeddings, context], dim=1)
        hidden, cell = self.lstm_cell(lstm_input, (hidden, cell))
        logits = self.fc(self.dropout(hidden))
        return logits, hidden, cell

    def forward(self, images, captions):
        encoder_out, hidden, cell = self.encode_image(images)
        logits = []
        seq_len = captions.size(1)
        for t in range(seq_len):
            step_logits, hidden, cell = self.decode_step(
                encoder_out, captions[:, t], hidden, cell
            )
            logits.append(step_logits.unsqueeze(1))
        return torch.cat(logits, dim=1)

    def freeze_encoder(self):
        for param in self.encoder.parameters():
            param.requires_grad = False

    def unfreeze_encoder_layer4(self):
        # Keep previous layers frozen
        for param in self.encoder.parameters():
            param.requires_grad = False
        # Unfreeze Layer 4 (index 7 in Sequential)
        for param in self.encoder[7].parameters():
            param.requires_grad = True
            
    def unfreeze_encoder_layer3(self):
        # Unfreeze Layer 3 (index 6) AND Layer 4 (index 7)
        for param in self.encoder[6].parameters():
            param.requires_grad = True
        for param in self.encoder[7].parameters():
            param.requires_grad = True

# Training

In [ ]:
def build_optimizer(net, encoder_lr, decoder_lr):
    encoder_params = [p for p in net.encoder.parameters() if p.requires_grad]
    decoder_params = [
        p for name, p in net.named_parameters()
        if p.requires_grad and not name.startswith('encoder.')
    ]
    param_groups = []
    if encoder_params:
        param_groups.append({'params': encoder_params, 'lr': encoder_lr})
    param_groups.append({'params': decoder_params, 'lr': decoder_lr})
    return torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

# Initialize criterion
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=LABEL_SMOOTHING)

def run_epoch(net, loader, optimizer=None, desc='epoch'):
    is_train = optimizer is not None
    net.train(is_train)
    total_loss = 0.0
    total_tokens = 0
    correct_tokens = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        # Correctly unpack 3 values from the CaptionDataset
        for images, captions, lengths in tqdm(loader, leave=False, desc=desc):
            images = images.to(DEVICE, non_blocking=True)
            captions = captions.to(DEVICE, non_blocking=True)

            # Teacher forcing: inputs are <start>...caption, targets are caption...<end>
            inputs = captions[:, :-1]
            targets = captions[:, 1:]

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = net(images, inputs)
                loss = criterion(
                    logits.reshape(-1, VOCAB_SIZE),
                    targets.reshape(-1),
                )

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

            mask = targets != PAD_IDX
            batch_tokens = mask.sum().item()
            total_loss += loss.item() * batch_tokens
            total_tokens += batch_tokens

            with torch.no_grad():
                preds = logits.argmax(dim=-1)
                correct_tokens += (preds[mask] == targets[mask]).sum().item()

    avg_loss = total_loss / max(total_tokens, 1)
    accuracy = correct_tokens / max(total_tokens, 1)
    return avg_loss, accuracy

In [ ]:
import math
import time
import torch

# 1. Update the loop to start exactly at Epoch 11
START_EPOCH = 16

# 2. Re-initialize the history tracker lists *only* if you want to clear old logs.
# If you want to keep appending to your existing history dictionary, 
# COMMENT OUT this history block so it doesn't overwrite your variables!
history = {
    'train_loss': [],
    'val_loss': [],
    'train_ppl': [],
    'val_ppl': [],
    'train_acc': [],
    'val_acc': [],
    'learning_rates': [],
}

print(
    f"{'Epoch':>6}  {'Tr Loss':>8}  {'Tr PPL':>7}  {'Tr Acc':>7}  "
    f"{'Val Loss':>9}  {'Val PPL':>8}  {'Val Acc':>8}  {'Time':>6}"
)
print('-' * 85)

# Notice we start the range from START_EPOCH now
for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    t0 = time.time()

    # --- MANUAL OVERRIDE FOR RESUME STEP ---
    if epoch == START_EPOCH:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-4
        print(f"\n>>> Resuming training. Manual Override: Set learning rate to {optimizer.param_groups[-1]['lr']:.2e}")
    # ---------------------------------------

    # This block handles layer unfreezing if your FREEZE_ENCODER_EPOCHS is set higher
    if epoch == FREEZE_ENCODER_EPOCHS + 1:
        model.unfreeze_encoder_layer4()
        optimizer = build_optimizer(model, ENCODER_LR, DECODER_LR)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6
        )
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'\n>>> Unfroze encoder layer4 — trainable params: {trainable:,}')

    # Standard training and validation steps using the existing model in RAM
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, desc=f'train {epoch}')
    val_loss, val_acc = run_epoch(model, val_loader, desc=f'val {epoch}')
    
    # Update the scheduler with the new validation loss
    scheduler.step(val_loss)

    train_ppl = math.exp(min(train_loss, 20))
    val_ppl = math.exp(min(val_loss, 20))
    current_lr = optimizer.param_groups[-1]['lr']
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_ppl'].append(train_ppl)
    history['val_ppl'].append(val_ppl)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['learning_rates'].append(current_lr)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
        saved = '✓'
    else:
        epochs_without_improvement += 1
        saved = ' '

    gap = val_loss - train_loss
    status = 'OK'
    if gap < -0.05:
        status = 'possible underfit'
    elif gap > 0.35:
        status = 'possible overfit'

    print(
        f'{epoch:>6}  {train_loss:>8.4f}  {train_ppl:>7.2f}  {train_acc:>6.2%}  '
        f'{val_loss:>9.4f}  {val_ppl:>8.2f}  {val_acc:>7.2%}  {elapsed:>5.0f}s {saved}  '
        f'gap={gap:+.4f} ({status}) lr={current_lr:.2e}'
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping triggered after {epoch} epochs.')
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f'\nRestored best checkpoint (val_loss={best_val_loss:.4f}).')

# Diagnostics & test evaluation

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(epochs_ran, history['train_loss'], marker='o', label='Train')
axes[0].plot(epochs_ran, history['val_loss'], marker='o', label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-entropy loss (per token)')
axes[0].set_title('Training progression')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history['train_ppl'], marker='o', label='Train')
axes[1].plot(epochs_ran, history['val_ppl'], marker='o', label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_ran, [a * 100 for a in history['train_acc']], marker='o', label='Train')
axes[2].plot(epochs_ran, [a * 100 for a in history['val_acc']], marker='o', label='Validation')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Token accuracy (%)')
axes[2].set_title('Token accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


def word_overlap_score(generated, references):
    generated_words = set(tokenize_caption(generated))
    best = 0.0
    for reference in references:
        reference_words = set(tokenize_caption(reference))
        if not reference_words:
            continue
        best = max(best, len(generated_words & reference_words) / len(reference_words))
    return best


test_loss, test_acc = run_epoch(model, test_loader, desc='test')
test_ppl = math.exp(min(test_loss, 20))
print(f'Test loss: {test_loss:.4f} | Test PPL: {test_ppl:.2f} | Test acc: {test_acc:.2%}')

reference_map = caption_df.groupby('image')['caption'].apply(list).to_dict()


In [ ]:
# BLEU-4 evaluation (standard metric for Flickr8k captioning)
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk'])
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

BLEU_EVAL_IMAGES = min(400, len(test_images))
eval_rng = random.Random(SEED)
bleu_images = sorted(test_images)
eval_rng.shuffle(bleu_images)
bleu_images = bleu_images[:BLEU_EVAL_IMAGES]

all_references = []
all_hypotheses = []
smoother = SmoothingFunction().method1

print(f'Computing BLEU-4 on {len(bleu_images)} test images (greedy decoding)...')
model.eval()

for i, image_name in enumerate(bleu_images):
    image_path = IMAGES_DIR / image_name
    image = eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    generated = generate_greedy(model, image, word2idx, idx2word)

    refs = [tokenize_caption(r) for r in reference_map[image_name]]
    hyp = tokenize_caption(generated)
    all_references.append(refs)
    all_hypotheses.append(hyp)

    if (i + 1) % 100 == 0:
        print(f'  {i + 1}/{len(bleu_images)} done...')

bleu4 = corpus_bleu(
    all_references,
    all_hypotheses,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smoother,
)
history['test_bleu4'] = bleu4
history['test_bleu4_n_images'] = len(bleu_images)
print(f'\nBLEU-4 on test set: {bleu4 * 100:.2f}')


In [ ]:
eval_images = sorted(test_images)
eval_rng = random.Random(SEED + 1)
eval_rng.shuffle(eval_images)
eval_images = eval_images[: min(200, len(eval_images))]

overlap_scores = []
for image_name in eval_images:
    image_path = IMAGES_DIR / image_name
    image = eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    generated = generate_greedy(model, image, word2idx, idx2word)
    overlap_scores.append(word_overlap_score(generated, reference_map[image_name]))

print(f'Mean word-overlap on {len(eval_images)} test images: {np.mean(overlap_scores):.3f}')

demo_images = eval_images[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, image_name in zip(axes.flatten(), demo_images):
    image_path = IMAGES_DIR / image_name
    image = eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    generated = generate_greedy(model, image, word2idx, idx2word)
    references = reference_map[image_name][:2]
    ax.imshow(Image.open(image_path).convert('RGB'))
    ax.axis('off')
    ax.set_title(
        f'Generated: {generated}\nRef: {" | ".join(references)}',
        fontsize=8,
    )
plt.suptitle('Test-set greedy captions', fontsize=14)
plt.tight_layout()
plt.show()


# Save

In [ ]:
checkpoint_path = MODEL_DIR / 'best_model.pt'
vocab_path = MODEL_DIR / 'vocabulary.json'
config_path = MODEL_DIR / 'config.json'
history_path = MODEL_DIR / 'training_history.json'
splits_path = MODEL_DIR / 'data_splits.json'

torch.save(model.state_dict(), checkpoint_path)

with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'word2idx': word2idx,
            'idx2word': {str(k): v for k, v in idx2word.items()},
        },
        f,
        indent=2,
    )

config = {
    'model_type': 'attention_lstm',
    'encoder_name': ENCODER_NAME,
    'encoder_dim': ENCODER_DIM,
    'embed_dim': EMBED_DIM,
    'hidden_dim': HIDDEN_DIM,
    'dropout': DROPOUT,
    'max_seq_len': MAX_SEQ_LEN,
    'vocab_size': VOCAB_SIZE,
    'pad_idx': PAD_IDX,
    'start_token': START_TOKEN,
    'end_token': END_TOKEN,
    'pad_token': PAD_TOKEN,
    'unk_token': UNK_TOKEN,
    'image_size': IMAGE_SIZE,
    'normalize_mean': IMAGENET_MEAN,
    'normalize_std': IMAGENET_STD,
    'eval_resize': IMAGE_SIZE + 32,
    'eval_center_crop': True,
}

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

with open(splits_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'seed': SEED,
            'train_images': sorted(train_images),
            'val_images': sorted(val_images),
            'test_images': sorted(test_images),
        },
        f,
        indent=2,
    )

print(f'Saved model       -> {checkpoint_path}')
print(f'Saved vocabulary  -> {vocab_path}')
print(f'Saved config      -> {config_path}')
print(f'Saved history     -> {history_path}')
print(f'Saved splits      -> {splits_path}')
if 'test_bleu4' in history:
    print(f'Test BLEU-4       -> {history["test_bleu4"] * 100:.2f}')
